# CME Futures: Equal-Weight Signal Backtests

This notebook sends every complete model configuration and checkpoint through the same signal
baseline. At each weekly decision, the signal ranks products by the selected prediction row and
holds equal-weight long and short groups for each configured concentration. This is
`stage='signal'`; equal weight is not an allocation method in the next stage.

Reader-facing prices and decisions use `product`. The shared boundary records the front-contract
position, raw-to-adjusted roll identity, cumulative-ratio transitions, expiry reference, contract
specifications, prediction lineage, and state-transition policy before converting `product` to
the existing engine's internal `symbol` key.

<!-- ML4T_ZH_BEGIN c0000-p000 -->
# CME 期貨：等權重訊號回測

本 Notebook 讓每個完整的模型設定與檢查點都通過相同的訊號基準。每次每週決策時，訊號依選定的預測資料列為商品排序，並針對每個設定的集中度，持有等權重的多頭與空頭群組。這是
`stage='signal'`；等權重不是下一階段的配置方法。

供讀者查看的價格與決策使用 `product`。共用邊界會記錄近月合約部位、原始價格與調整後價格之間的轉倉識別資訊、累積比率的轉換、到期日參照、合約規格、預測資料沿革及狀態轉換政策，然後才將 `product` 轉換為既有引擎內部的 `symbol` 鍵值。
<!-- ML4T_ZH_END c0000-p000 -->


## What this stage is for, and what it deliberately does not do

A model that predicts returns well is not yet a strategy, and the gap between the two is where
most of the disappointment in quantitative investing lives. A prediction is a number attached to
a product and a date. Turning it into a position requires deciding how many products to hold,
how much of each, when to change, and what that changing costs - and each of those decisions can
destroy a real edge or manufacture a fake one.

This stage answers only the first of those questions, and answers it in the plainest way
available. At each weekly decision the products are ranked by their predicted return, the top
`k` are held long, the bottom `k` short, and every position in a leg is the same size. Nothing
is optimized. There is no covariance matrix, no risk target, no position limit, no cost model.

The plainness is the point. This is the **baseline** every later stage is measured against, so
it has to be a construction whose behaviour comes from the predictions and from nothing else.
When the portfolio-construction stage reports a higher Sharpe, the question a reader should be
able to ask is "higher than what?" - and the answer has to be a number that no modelling choice
of ours is hiding inside.

Two consequences follow, and both are easy to misread later:


<!-- ML4T_ZH_BEGIN c0001-p000 -->
## 本階段的用途，以及刻意不處理的事項

能準確預測報酬率的模型，還不能算是一套策略；兩者之間的落差，正是量化投資中大多數失望的來源。預測只是附在某項商品與某個日期上的一個數字。要把預測轉成部位，必須決定持有多少種商品、各持有多少、何時調整，以及調整要付出多少成本；其中任何一項決策都可能摧毀真正的優勢，或製造虛假的優勢。

本階段只回答其中第一個問題，並採用最直白的方式。每次每週決策時，依預測報酬率為商品排序，買進排名前 `k` 的商品，放空排名後 `k` 的商品，同一側的每個部位規模相同。不做任何最佳化；沒有共變異數矩陣、風險目標、部位上限或成本模型。

這種簡單性正是重點。這是衡量後續每個階段的**基準**，因此其表現必須只來自預測，而不受其他因素影響。當投資組合建構階段報告較高的夏普比率時，讀者應能問「比什麼高？」；答案必須是一個未藏入任何我們的建模選擇的數字。

由此可得兩個結論，往後都很容易被誤讀：
<!-- ML4T_ZH_END c0001-p000 -->

- **Equal weight here is not an allocation method.** It is the absence of one. The next stage's
  allocation methods are compared against this, and one of them being equal-weight-like is a
  result about that stage rather than a repetition of this one.
- **These Sharpe ratios are not the case study's results.** They are the reference the results
  are quoted against. Reporting one of them as the strategy's performance would be quoting the
  control arm as the finding.

<!-- ML4T_ZH_BEGIN c0001-p001 -->
- **此處的等權重不是配置方法。** 它代表尚未採用配置方法。下一階段的配置方法會與此比較；即使其中一種方法類似等權重，那也是該階段的結果，而不是本階段的重複。
- **這些夏普比率不是本案例研究的結果。** 它們是呈現結果時的比較基準。把其中一個當作策略績效報告，等於把對照組當成研究發現。
<!-- ML4T_ZH_END c0001-p001 -->


In [1]:
"""Run the complete CME futures equal-weight validation baseline."""

import polars as pl

from case_studies.cme_futures.research_workflow import (
    ALL_LABELS,
    MODEL_POPULATION_NAMES,
    create_label_candidate_sets,
    load_futures_price_path,
    official_prediction_catalog,
    open_study,
    preview_prediction_candidates,
    run_official_backtest_requests,
    strategy_request_frame,
)
from case_studies.research.population import supersedes_for_run
from case_studies.utils.sweep_config import get_top_k_values_for

In [2]:
EXECUTION_TIER = "canonical"
WORKSPACE: str | None = None
PREVIEW_LABELS: list[str] = []
PREVIEW_MAX_PREDICTIONS = 0

# The baseline population is immutable under its name, so a run whose members have moved has to
# say which generation it retires. Anything upstream that changes a backtest identity moves them:
# a corrected label, a changed accounting field, or a re-run after a registry reset all produce a
# different member list under the same name, and `OfficialPopulation.create` refuses to write it
# without being told what it replaces. Declared here as a literal so that running the committed
# notebook as it stands recomputes the population on record. Empty for a first snapshot.
BASELINE_POPULATION = "cme_futures-signal-validation-v1"
SUPERSEDES_BASELINE_POPULATION: str = ""

# The per-label candidate sets this notebook freezes are immutable under their names too, and
# for the same reason as the population above: `CandidateSet.create` refuses a changed member
# list under a name that already exists. Nothing reached that argument before, so any run whose
# membership moved - which a wider sweep does by construction - stopped at the freeze after the
# fit, with no parameter able to answer it.
#
# Each name maps to the generation this run retires. `"live"` names the lineage and looks the
# generation up, which is the form that does not decay: naming the head instead is correct only
# until the next publish, because `create` accepts the head and nothing else. The declaration is
# resolved through `candidate_set_supersedes` rather than offered straight, so a reader's clean
# clone - which has no generation to replace, and often no `candidate_sets` table at all -
# publishes generation one instead of being refused. An unchanged re-run never reads it: a set's
# hash is computed from its members and its contract, so the existing name binding answers.
SUPERSEDES_CANDIDATE_SETS: dict[str, str] = {
    "cme_futures-signal-fwd_ret_5d-v1": "live",
    "cme_futures-signal-fwd_ret_21d-v1": "live",
}

## Futures data used by the strategy

The model predicts a continuous front-contract return. `raw_close` is the traded contract level;
`adj_close` is the multiplicatively back-adjusted level used for continuous returns. A change in
`cum_ratio` identifies a roll transition. The backtest consumes adjusted OHLC while retaining this
audit table and the product expiry rules in its identity.

<!-- ML4T_ZH_BEGIN c0004-p000 -->
## 策略使用的期貨資料

模型預測連續近月合約的報酬率。`raw_close` 是實際交易合約的價格水準；
`adj_close` 是用於計算連續報酬率、經乘法反向調整的價格水準。`cum_ratio` 的變化代表一次轉倉。回測使用調整後的開高低收價格，同時在其識別資訊中保留這份稽核表與商品到期規則。
<!-- ML4T_ZH_END c0004-p000 -->


In [3]:
study = open_study(execution_tier=EXECUTION_TIER, workspace=WORKSPACE)
if EXECUTION_TIER == "canonical":
    if PREVIEW_LABELS or PREVIEW_MAX_PREDICTIONS:
        raise ValueError("canonical execution cannot declare preview reductions")
    labels = ALL_LABELS
elif EXECUTION_TIER == "preview":
    if WORKSPACE is None or not PREVIEW_LABELS or PREVIEW_MAX_PREDICTIONS < 1:
        raise ValueError(
            "preview execution requires WORKSPACE, PREVIEW_LABELS and PREVIEW_MAX_PREDICTIONS"
        )
    unknown = sorted(set(PREVIEW_LABELS) - set(ALL_LABELS))
    if unknown:
        raise ValueError(f"preview labels this case study does not declare: {unknown}")
    labels = tuple(PREVIEW_LABELS)
else:
    raise ValueError(f"unsupported execution tier: {EXECUTION_TIER!r}")
price_paths = {label: load_futures_price_path(label) for label in labels}
market_rows = []
for label, path in price_paths.items():
    roll_counts = path.roll_transitions.group_by("product").len().rename({"len": "rolls"})
    market_rows.append(
        path.audit.group_by("product")
        .agg(
            pl.col("timestamp").min().alias("first_session"),
            pl.col("timestamp").max().alias("last_session"),
        )
        .join(roll_counts, on="product", how="left")
        .join(path.expiry_rules, on="product", how="left")
        .with_columns(pl.lit(label).alias("label"), pl.col("rolls").fill_null(0))
    )
market_contract = pl.concat(market_rows).sort("label", "product")

In [4]:
market_contract

product,first_session,last_session,rolls,expiry_rule,contract_months,label
str,datetime[ms],datetime[ms],u32,str,str,str
"""6A""",2019-01-03 00:00:00,2023-11-29 00:00:00,19,"""2bd_before_3rd_wednesday""","""H,M,U,Z""","""fwd_ret_21d"""
"""6B""",2019-01-03 00:00:00,2023-11-29 00:00:00,18,"""2bd_before_3rd_wednesday""","""H,M,U,Z""","""fwd_ret_21d"""
"""6C""",2019-01-03 00:00:00,2023-11-29 00:00:00,19,"""2bd_before_3rd_wednesday""","""H,M,U,Z""","""fwd_ret_21d"""
"""6E""",2019-01-03 00:00:00,2023-11-29 00:00:00,19,"""2bd_before_3rd_wednesday""","""H,M,U,Z""","""fwd_ret_21d"""
"""6J""",2019-01-03 00:00:00,2023-11-29 00:00:00,19,"""2bd_before_3rd_wednesday""","""H,M,U,Z""","""fwd_ret_21d"""
…,…,…,…,…,…,…
"""ZM""",2019-01-03 00:00:00,2023-12-21 00:00:00,45,"""business_day_before_15th""","""F,H,K,N,Q,U,V,Z""","""fwd_ret_5d"""
"""ZN""",2019-01-03 00:00:00,2023-12-21 00:00:00,20,"""last_business_day""","""H,M,U,Z""","""fwd_ret_5d"""
"""ZS""",2019-01-03 00:00:00,2023-12-21 00:00:00,26,"""business_day_before_15th""","""F,H,K,N,Q,U,X""","""fwd_ret_5d"""


## Complete baseline requests

The source rows come from the six official model populations. Each population must be complete
before this cell can construct a request. No registry ordering, row cap, cached metric, or caught
failure can remove a candidate.

**Why completeness is enforced rather than assumed.** A backtest sweep that silently skipped a
configuration would still produce a leaderboard, and the leaderboard would still look sensible.
What it would no longer support is the comparison it exists for: selecting the best validation
Sharpe out of a population means nothing if the population is whatever happened to finish. The
failure mode is not a wrong number, it is a right-looking number computed over a set nobody can
reconstruct - so the completeness check runs before any request is built rather than after.

**What a request is.** One row here is one backtest to run: a prediction set identified by its
hash, the label it was fitted against, and a signal specification. The `allocation`, `risk` and
`costs` fields are all None, which is what makes these the baseline - later chapters fill them
in and re-run this same machinery.


<!-- ML4T_ZH_BEGIN c0007-p000 -->
## 完整的基準回測請求

來源資料列取自六個官方模型群體。每個群體都必須完整，這個儲存格才能建立請求。登錄資料庫的排序、資料列數量上限、快取指標或被捕捉的失敗，都不能移除任何候選設定。

**為何要強制檢查完整性，而不是假定它已完整。** 若回測掃描悄悄略過一項設定，仍會產生排行榜，且看起來仍然合理。但它已無法支持原本要做的比較：如果群體只是碰巧完成的設定，從中選出驗證期夏普比率最高者便毫無意義。失敗之處不是數字錯誤，而是根據無人能重建的集合算出一個看似正確的數字。因此，完整性檢查會在建立任何請求之前執行，而非事後才做。

**請求的意義。** 此處一列代表一次待執行的回測：以雜湊值識別的預測集、模型擬合所依據的標籤，以及訊號規格。`allocation`、`risk` 與 `costs` 欄位全為 None，這正是它們成為基準的原因；後續章節會填入這些欄位，再以相同機制重新執行。
<!-- ML4T_ZH_END c0007-p000 -->

**Why several `top_k` values rather than one.** `top_k` is how many products each leg holds, and
it is the one dial this stage does turn. It decides concentration, and concentration trades two
things against each other: a small `k` puts more weight behind the predictions the model is most
confident about, and a large `k` averages across more of them so a single product's idiosyncratic
move matters less. Which wins is a property of the signal's strength and of how quickly its
ranking decays, neither of which is known before running it. Sweeping the values that
`get_top_k_values_for` derives from the tradeable universe answers the question with the data
rather than by picking a round number, and it means a configuration that only works at one
concentration is visible as such rather than being represented by its best case.

<!-- ML4T_ZH_BEGIN c0007-p001 -->
**為何使用多個 `top_k` 值，而非只有一個。** `top_k` 是每一側持有的商品數，也是本階段唯一調整的參數。它決定集中度，而集中度涉及兩者取捨：較小的 `k` 讓模型最有把握的預測占較大權重；較大的 `k` 則分散到更多商品，使單一商品的特有波動影響較小。何者較好，取決於訊號強度及其排序衰減的速度，而這兩者在執行前都無從得知。掃描由 `get_top_k_values_for` 根據可交易商品範圍推導出的數值，便能用資料回答問題，而非任選一個整數；這也讓只在單一集中度下奏效的設定如實呈現，不會只以其最佳情況代表整體。
<!-- ML4T_ZH_END c0007-p001 -->


In [5]:
if EXECUTION_TIER == "canonical":
    predictions = official_prediction_catalog(study, MODEL_POPULATION_NAMES)
else:
    predictions = preview_prediction_candidates(study, labels=labels, limit=PREVIEW_MAX_PREDICTIONS)
request_rows = []
for label in labels:
    label_catalog = predictions.filter(pl.col("label") == label)
    n_products = price_paths[label].prices.get_column("product").n_unique()
    for row in label_catalog.iter_rows(named=True):
        for top_k in get_top_k_values_for("cme_futures", label, n_products):
            request_rows.append(
                {
                    "request_name": f"{row['prediction_hash']}-equal-weight-k{top_k}",
                    "prediction_hash": row["prediction_hash"],
                    "label": label,
                    "signal": {"method": "equal_weight_top_k", "top_k": top_k},
                    "allocation": None,
                    "risk": None,
                    "costs": None,
                    "chapter": "ch16",
                }
            )
requests = strategy_request_frame(request_rows)
requests.select("request_name", "prediction_hash", "label", "signal")

request_name,prediction_hash,label,signal
str,str,str,object
"""8ef2b9b47e4c-equal-weight-k5""","""8ef2b9b47e4c""","""fwd_ret_5d""","{'method': 'equal_weight_top_k', 'top_k': 5}"
"""8ef2b9b47e4c-equal-weight-k10""","""8ef2b9b47e4c""","""fwd_ret_5d""","{'method': 'equal_weight_top_k', 'top_k': 10}"
"""e8f8cbe1eee4-equal-weight-k5""","""e8f8cbe1eee4""","""fwd_ret_5d""","{'method': 'equal_weight_top_k', 'top_k': 5}"
"""e8f8cbe1eee4-equal-weight-k10""","""e8f8cbe1eee4""","""fwd_ret_5d""","{'method': 'equal_weight_top_k', 'top_k': 10}"
"""04c977889406-equal-weight-k5""","""04c977889406""","""fwd_ret_5d""","{'method': 'equal_weight_top_k', 'top_k': 5}"
…,…,…,…
"""c1a637d80ab3-equal-weight-k10""","""c1a637d80ab3""","""fwd_ret_21d""","{'method': 'equal_weight_top_k', 'top_k': 10}"
"""0d37e2de349a-equal-weight-k5""","""0d37e2de349a""","""fwd_ret_21d""","{'method': 'equal_weight_top_k', 'top_k': 5}"
"""0d37e2de349a-equal-weight-k10""","""0d37e2de349a""","""fwd_ret_21d""","{'method': 'equal_weight_top_k', 'top_k': 10}"


## Execute and freeze the comparable sets

Target weights are canonical typed decisions with unique `product,timestamp` keys and exact
prediction eligibility. Expected backtest identities are snapshotted before the engine runs.
Every member must finish before the per-label validation candidate sets are created.

### A population is named, and a name means one thing

The results of this sweep are written into the registry as an **official population**: a named,
frozen list of exactly which backtests belong to the comparison. Downstream notebooks select
from a population by name, so the name has to keep meaning the same set - otherwise a selection
made last month and a selection made today would be answering different questions while
appearing to answer the same one.

That is why the registry refuses to write a different member list under a name that already
exists. It is also why a re-run has to say what it retires. Anything that moves a backtest
identity moves the members: a corrected label, a changed accounting field, a re-run after a
registry reset. `SUPERSEDES_BASELINE_POPULATION` in the parameter cell is where that is
declared, and it names the generation this run replaces rather than deleting it - the retired
snapshot stays in the registry, so a result quoted from it remains traceable to the population
it was actually computed over.

A preview run publishes no population at all. It is discarded with its workspace, has no
lineage to extend, and offering a supersession from one would retire a canonical generation in
favour of something nobody kept.

### What happens at a fold boundary, and what it costs to read


<!-- ML4T_ZH_BEGIN c0009-p000 -->
## 執行並封存可比較的集合

目標權重是具有標準型別的決策，採用唯一的 `product,timestamp` 鍵值，且預測資格必須完全吻合。在引擎執行前，先擷取預期回測識別資訊的快照。所有成員都必須完成，才能建立依標籤區分的驗證期候選集合。

### 群體有名稱，而名稱只能代表一個集合

這次掃描的結果會以**官方群體**寫入登錄資料庫：一份具名且已封存的清單，明確列出哪些回測屬於這次比較。後續 Notebook 會依名稱從群體中選取，因此名稱必須始終指向同一集合；否則上個月與今天做出的選擇，看似回答同一問題，實際上卻回答不同問題。

因此，登錄資料庫拒絕在既有名稱下寫入不同的成員清單。這也是重新執行時必須聲明要汰換哪一版的原因。任何改變回測識別資訊的事項都會改變成員，例如修正後的標籤、變更的會計欄位，或登錄資料庫重設後的重新執行。參數儲存格中的 `SUPERSEDES_BASELINE_POPULATION` 用來聲明此事；它指定本次執行取代的版本，而非刪除該版本。已汰換的快照仍留在登錄資料庫，因此引用其結果時，依然能追溯到實際計算所依據的群體。

預覽執行完全不會發布群體。預覽會連同工作區一併捨棄，沒有可延續的資料沿革；若允許用預覽來指定取代版本，就會讓無人保存的結果取代正式版本。

### 驗證折邊界會發生什麼事，以及解讀結果的代價
<!-- ML4T_ZH_END c0009-p000 -->

The five validation folds are consecutive stretches of calendar time, and this backtest runs
through them as one series of weekly decisions rather than as five separate simulations. At a
boundary the position **carries**; it is not flattened. The declared policy is
`StateTransitionPolicy(fold_boundary="continue")`.

Two reasons, and the second is the harder one. Nothing happens in the market on the four dates
that separate the folds - they are an index this case study cut for evaluation, not events - so
flattening there would be an artifact of how the sample was divided. And the liquidation could
not be executed here in any case: the schedule decides on Friday's close and fills at Monday's
open, so there is no weight row for the engine to snap a reset onto, and it refuses to snap one
forward rather than carry the old state across the boundary and then charge a round trip for no
change in exposure.

**So a per-fold number in this pipeline is not computed from a flat start.** A fold inherits at
most one week of exposure from the fold before it. That is four of roughly 260 weekly decisions,
about 1.5% of them, and it is the reason not to read a per-fold Sharpe here or downstream as
though the fold were a standalone track record. The alternative - same-bar execution, which would
buy the flat start - is what makes a futures backtest implausible, and it is not a trade worth
making for four decisions.

<!-- ML4T_ZH_BEGIN c0009-p001 -->
五個驗證折是日曆時間上連續的區間；本回測將它們視為一系列每週決策執行，而不是五次獨立模擬。在邊界處，部位會**延續持有**，不會全部平倉。聲明的政策為 `StateTransitionPolicy(fold_boundary="continue")`。

原因有兩個，第二個更棘手。分隔各折的四個日期，市場上並未發生任何特殊事件；它們只是本案例研究為評估而切分樣本的索引，因此在那些日期平倉會是樣本切分方式造成的假象。而且此處根本無法執行該清算：排程在週五收盤時決策，於週一開盤成交，因此沒有可讓引擎對齊重設的權重資料列；引擎拒絕將重設順延，而會讓舊狀態跨過邊界繼續持有，避免在曝險未變時仍收取一次來回交易成本。

**因此，這條流程中的單一折數值不是從空手狀態開始計算。** 每一折最多承接前一折一週的曝險。相對於約 260 次每週決策，這樣的情況只有四次，約占 1.5%。因此，無論在此處或後續階段，都不能把單一折的夏普比率當成獨立的績效紀錄來解讀。另一種做法是在同一根價格棒內執行，以取得空手起點，但那會使期貨回測不可信；為了四次決策而付出這種代價並不值得。
<!-- ML4T_ZH_END c0009-p001 -->


In [6]:
execution = run_official_backtest_requests(
    study,
    requests,
    population_name=BASELINE_POPULATION if EXECUTION_TIER == "canonical" else None,
    supersedes=supersedes_for_run(
        study,
        population_name=BASELINE_POPULATION,
        declared=SUPERSEDES_BASELINE_POPULATION or None,
        execution_tier=EXECUTION_TIER,
    ),
)
# A candidate set is canonical too - `CandidateSet.create` refuses a preview member
# (research/comparison.py:50-51) - so a preview run leaves the funnel's named pools alone and
# the notebooks downstream read its backtest catalog directly instead.
candidate_sets = (
    create_label_candidate_sets(
        study, execution, stage="signal", supersedes_by_set=SUPERSEDES_CANDIDATE_SETS
    )
    if EXECUTION_TIER == "canonical"
    else {}
)

`source` says whether each member was computed by this run or served from the registry because
an identical identity was already recorded. A re-run of a registered sweep is entirely `reused`
and completes in seconds; without the column that is indistinguishable from having computed
every row.

<!-- ML4T_ZH_BEGIN c0011-p000 -->
`source` 表示每個成員是本次執行新計算的，還是因為相同識別資訊已記錄在登錄資料庫，而直接取用既有結果。重新執行已登錄的掃描時，所有成員都會是 `reused`，幾秒內即可完成；若沒有這一欄，就無法區分它與重新計算了每一列的情況。
<!-- ML4T_ZH_END c0011-p000 -->


In [7]:
execution.catalog_rows.sort("label", "request_name")

request_name,label,prediction_hash,decision_hash,backtest_hash,complete,source
str,str,str,str,str,bool,str
"""0091960d1774-equal-weight-k10""","""fwd_ret_21d""","""0091960d1774""","""51694f3562d0""","""6797da4a73fe""",true,"""computed"""
"""0091960d1774-equal-weight-k5""","""fwd_ret_21d""","""0091960d1774""","""f7394b11c563""","""d89b93ae20c6""",true,"""computed"""
"""02ccea35b5ff-equal-weight-k10""","""fwd_ret_21d""","""02ccea35b5ff""","""02fadf2ff25d""","""27c7870514ab""",true,"""computed"""
"""02ccea35b5ff-equal-weight-k5""","""fwd_ret_21d""","""02ccea35b5ff""","""f27b9ccf9aae""","""8dda80515e6a""",true,"""computed"""
"""043c2027c0a2-equal-weight-k10""","""fwd_ret_21d""","""043c2027c0a2""","""e67ad8200117""","""0cbb3d6807ba""",true,"""computed"""
…,…,…,…,…,…,…
"""ff0133b4e8fb-equal-weight-k5""","""fwd_ret_5d""","""ff0133b4e8fb""","""4a9fd7aa0dfe""","""2eaa7fdd6ac0""",true,"""computed"""
"""ffd902794a3e-equal-weight-k10""","""fwd_ret_5d""","""ffd902794a3e""","""7a7565f20eca""","""7a795511e5c5""",true,"""computed"""
"""ffd902794a3e-equal-weight-k5""","""fwd_ret_5d""","""ffd902794a3e""","""2c024e9a93ee""","""53768131a7ee""",true,"""computed"""


`14_portfolio_management` ranks each immutable per-label set by validation backtest Sharpe.
`19_strategy_analysis` interprets the validated strategy results.

<!-- ML4T_ZH_BEGIN c0013-p000 -->
`14_portfolio_management` 依驗證期回測夏普比率，為每個不可變的依標籤區分集合排序。
`19_strategy_analysis` 解讀通過驗證的策略結果。
<!-- ML4T_ZH_END c0013-p000 -->
